# agentic_alpha_ml_pair_search_v1

Diagnostic R-G-E-V-D card for two external-knowledge branches that were not closed by the earlier cards: a walk-forward crash/return ensemble using qlib SOXL/SOXX/VIX features, and a SOXL/SOXX pair hedge inspired by leveraged ETF volatility, trend, and hedge research. The goal gate remains strict: beat `production/qlib_paper_sim_v1` return while keeping MaxDD above -20%.


In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from lightgbm import LGBMClassifier, LGBMRegressor
from qlib.data import D
from qlib.workflow import R
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_ml_pair_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
START = '2019-01-02'
END = '2026-06-09'
MAX_DRAWDOWN_FLOOR = -20.0
TRANSACTION_COST = 0.0005

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = f"""experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  train_start: 2011-03-10
  test:
    start: {START}
    end: {END}
rgevd:
  branch: ml_crash_return_and_pair_hedge
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
"""
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use research-generate-evaluate-validate-distill loops and close each branch with evidence.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'paper',
            'title': 'Leverage for the Long Run',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2741701',
            'extracted_signal': 'Levered exposure needs trend filters; high return without crash avoidance is not sufficient.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Volatility Targeting Is Trendy',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4773781',
            'extracted_signal': 'Volatility-managed returns overlap with trend-following, so the search combines return and crash scores.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Leveraged ETFs in Low-Volatility Environments',
            'url': 'https://quantpedia.com/leveraged-etfs-in-low-volatility-environments/',
            'extracted_signal': 'Realized-vs-implied volatility filters can identify favorable LETF exposure windows.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'arxiv',
            'title': 'Compounding Effects in Leveraged ETFs',
            'url': 'https://arxiv.org/html/2504.20116v1',
            'extracted_signal': 'LETF compounding depends on persistent trend and volatility clustering.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'twitter',
            'title': 'SOXL volatility and drawdown risk discussion',
            'url': 'https://x.com/Alpha_Ex_LLC/status/2067644619462938886',
            'extracted_signal': 'SOXL risk controls should react to volatility and drawdown state rather than raw signal only.',
            'retrieval_status': 'search_snippet_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')

qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
raw = D.features(
    ['SOXL', 'SOXX'],
    ['$open', '$high', '$low', '$close', '$volume', '$vix'],
    start_time='2010-03-11',
    end_time='2026-06-10',
    freq='day',
)
wide = raw.reset_index().pivot(index='datetime', columns='instrument')
wide.columns = [f'{field.strip("$")}_{instrument}' for field, instrument in wide.columns]
wide = wide.sort_index()

soxl_close = wide['close_SOXL']
soxx_close = wide['close_SOXX']
vix = wide['vix_SOXL']
volume = wide['volume_SOXL']
soxl_ret = soxl_close.pct_change()
soxx_ret = soxx_close.pct_change()
next_soxl_ret = soxl_ret.shift(-1)
next_soxx_ret = soxx_ret.shift(-1)

feature_parts: list[pd.DataFrame] = []
for window in [1, 2, 3, 5, 10, 21, 22, 40, 50, 63, 84, 126, 150, 200, 252]:
    payload = {
        f'ret{window}': soxl_close.pct_change(window),
        f'soxx_ret{window}': soxx_close.pct_change(window),
        f'sma{window}': soxl_close.rolling(window).mean(),
        f'soxx_sma{window}': soxx_close.rolling(window).mean(),
        f'ma{window}': soxl_close / soxl_close.rolling(window).mean() - 1,
        f'soxx_ma{window}': soxx_close / soxx_close.rolling(window).mean() - 1,
        f'dd{window}': soxl_close / soxl_close.rolling(window).max() - 1,
        f'volchg{window}': volume / volume.rolling(window).mean() - 1,
    }
    if window >= 2:
        payload[f'vol{window}'] = soxl_ret.rolling(window).std() * np.sqrt(252)
        payload[f'soxx_vol{window}'] = soxx_ret.rolling(window).std() * np.sqrt(252)
    feature_parts.append(pd.DataFrame(payload, index=wide.index))
features = pd.concat(feature_parts, axis=1)
features['vix'] = vix
features['vix_chg1'] = vix.pct_change()
features['vix_chg5'] = vix.pct_change(5)
for window in [21, 40, 63, 126, 252]:
    features[f'vix_z{window}'] = (vix - vix.rolling(window).mean()) / vix.rolling(window).std()
    features[f'vol_vix_ratio{window}'] = features[f'vol{window}'] / (vix / 100.0)
    features[f'soxx_vol_vix_ratio{window}'] = features[f'soxx_vol{window}'] / (vix / 100.0)
features['label_ret'] = next_soxl_ret
features['label_crash'] = (next_soxl_ret < -0.06).astype(int)
features['label_bigup'] = (next_soxl_ret > 0.04).astype(int)
features = features.replace([np.inf, -np.inf], np.nan).dropna()
feature_cols = [col for col in features.columns if not col.startswith('label_')]

prediction_frames: list[pd.DataFrame] = []
for year in range(2019, 2027):
    test_start = pd.Timestamp(f'{year}-01-01')
    test_end = pd.Timestamp(f'{year}-12-31') if year < 2026 else pd.Timestamp(END)
    train = features.loc[: test_start - pd.Timedelta(days=1)]
    test = features.loc[(features.index >= test_start) & (features.index <= test_end)]
    if len(train) < 500 or test.empty:
        continue
    x_train = train[feature_cols]
    x_test = test[feature_cols]
    ret_models = [
        LGBMRegressor(n_estimators=120, learning_rate=0.04, max_depth=3, num_leaves=15, min_child_samples=25, subsample=0.8, colsample_bytree=0.8, random_state=year, verbosity=-1),
        HistGradientBoostingRegressor(max_iter=120, learning_rate=0.04, max_leaf_nodes=15, l2_regularization=0.03, random_state=year),
        make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    ]
    crash_models = [
        LGBMClassifier(n_estimators=120, learning_rate=0.04, max_depth=3, num_leaves=15, min_child_samples=25, subsample=0.8, colsample_bytree=0.8, random_state=year + 1, verbosity=-1),
        HistGradientBoostingClassifier(max_iter=120, learning_rate=0.04, max_leaf_nodes=15, l2_regularization=0.03, random_state=year + 1),
        make_pipeline(StandardScaler(), LogisticRegression(C=0.25, max_iter=1000, class_weight='balanced')),
    ]
    up_models = [
        LGBMClassifier(n_estimators=120, learning_rate=0.04, max_depth=3, num_leaves=15, min_child_samples=25, subsample=0.8, colsample_bytree=0.8, random_state=year + 2, verbosity=-1),
        HistGradientBoostingClassifier(max_iter=120, learning_rate=0.04, max_leaf_nodes=15, l2_regularization=0.03, random_state=year + 2),
    ]
    ret_predictions = []
    crash_predictions = []
    up_predictions = []
    for model in ret_models:
        model.fit(x_train, train['label_ret'])
        ret_predictions.append(model.predict(x_test))
    for model in crash_models:
        model.fit(x_train, train['label_crash'])
        crash_predictions.append(model.predict_proba(x_test)[:, 1])
    for model in up_models:
        model.fit(x_train, train['label_bigup'])
        up_predictions.append(model.predict_proba(x_test)[:, 1])
    out = pd.DataFrame(index=test.index)
    out['ret_next'] = test['label_ret']
    out['ret_score'] = np.mean([pd.Series(pred, index=test.index).rank(pct=True).to_numpy() for pred in ret_predictions], axis=0)
    out['crash_score'] = np.mean(crash_predictions, axis=0)
    out['up_score'] = np.mean(up_predictions, axis=0)
    prediction_frames.append(out)
ml_predictions = pd.concat(prediction_frames).sort_index()

ml_returns = ml_predictions['ret_next'].to_numpy(dtype=float)

def evaluate_single_asset(name: str, exposure: np.ndarray) -> dict[str, float | str]:
    exposure = np.asarray(exposure, dtype=float)
    returns = exposure * ml_returns - np.abs(np.diff(np.r_[0.0, exposure])) * TRANSACTION_COST
    if np.any(returns <= -0.99):
        return {}
    equity = np.cumprod(1 + returns)
    drawdown = equity / np.maximum.accumulate(equity) - 1
    return {
        'candidate': name,
        'return_pct': float((equity[-1] - 1) * 100),
        'max_drawdown_pct': float(drawdown.min() * 100),
        'sharpe': float(returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else np.nan,
        'mean_exposure': float(exposure.mean()),
        'min_exposure': float(exposure.min()),
        'max_exposure': float(exposure.max()),
        'turnover': float(np.mean(np.abs(np.diff(np.r_[0.0, exposure])))),
    }

ml_rows = []
ret_score = ml_predictions['ret_score'].to_numpy(dtype=float)
crash_score = ml_predictions['crash_score'].to_numpy(dtype=float)
up_score = ml_predictions['up_score'].to_numpy(dtype=float)
for return_q in [0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.93, 0.95]:
    return_threshold = np.quantile(ret_score, return_q)
    for crash_q in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
        crash_threshold = np.quantile(crash_score, crash_q)
        for up_q in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
            up_threshold = np.quantile(up_score, up_q)
            long_gate = (ret_score > return_threshold) & (crash_score < crash_threshold) & (up_score > up_threshold)
            for long_exposure in [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0]:
                for neutral_exposure in [0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]:
                    exposure = np.where(long_gate, long_exposure, neutral_exposure)
                    row = evaluate_single_asset('walk_forward_crash_return_ensemble', exposure)
                    if row:
                        row.update({'branch': 'walk_forward_crash_return_ensemble', 'return_q': return_q, 'crash_q': crash_q, 'up_q': up_q, 'long_exposure': long_exposure, 'neutral_exposure': neutral_exposure})
                        ml_rows.append(row)
ml_grid = pd.DataFrame(ml_rows)

pair_data = pd.concat(
    [
        features[['sma21', 'sma22', 'sma40', 'sma50', 'sma63', 'sma84', 'sma150', 'sma200', 'sma252', 'ma150', 'ma200', 'ma252', 'ret21', 'ret22', 'ret40', 'ret50', 'ret63', 'ret84', 'vol40', 'vol_vix_ratio40', 'vix_z21']],
        next_soxl_ret.rename('soxl_next'),
        next_soxx_ret.rename('soxx_next'),
    ],
    axis=1,
).loc[START:END].dropna(subset=['soxl_next', 'soxx_next'])
pair_cols = {col: pair_data[col].to_numpy(dtype=float) for col in pair_data.columns if col not in {'soxl_next', 'soxx_next'}}
pair_soxl_returns = pair_data['soxl_next'].to_numpy(dtype=float)
pair_soxx_returns = pair_data['soxx_next'].to_numpy(dtype=float)

def evaluate_pair(name: str, soxl_exposure: np.ndarray, soxx_exposure: np.ndarray) -> dict[str, float | str]:
    soxl_exposure = np.asarray(soxl_exposure, dtype=float)
    soxx_exposure = np.asarray(soxx_exposure, dtype=float)
    returns = soxl_exposure * pair_soxl_returns + soxx_exposure * pair_soxx_returns - (np.abs(np.diff(np.r_[0.0, soxl_exposure])) + np.abs(np.diff(np.r_[0.0, soxx_exposure]))) * TRANSACTION_COST
    if np.any(returns <= -0.99):
        return {}
    equity = np.cumprod(1 + returns)
    drawdown = equity / np.maximum.accumulate(equity) - 1
    return {
        'candidate': name,
        'return_pct': float((equity[-1] - 1) * 100),
        'max_drawdown_pct': float(drawdown.min() * 100),
        'sharpe': float(returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else np.nan,
        'mean_soxl_exposure': float(soxl_exposure.mean()),
        'mean_soxx_exposure': float(soxx_exposure.mean()),
        'min_soxl_exposure': float(soxl_exposure.min()),
        'max_soxl_exposure': float(soxl_exposure.max()),
        'min_soxx_exposure': float(soxx_exposure.min()),
        'max_soxx_exposure': float(soxx_exposure.max()),
        'turnover': float(np.mean(np.abs(np.diff(np.r_[0.0, soxl_exposure])) + np.abs(np.diff(np.r_[0.0, soxx_exposure])))),
    }

pair_rows = []
base_params = [(84, 252, 'vol40', 0.4), (50, 252, 'vol40', 0.4), (40, 252, 'vol40', 0.4), (63, 252, 'vol40', 0.4), (22, 200, 'vol40', 0.4), (22, 200, 'vol_vix_ratio40', 0.4), (84, 252, 'vol_vix_ratio40', 0.4), (40, 252, 'vol_vix_ratio40', 0.4)]
for fast, slow, vol_col, vol_q in base_params:
    risk_on = (pair_cols[f'sma{fast}'] > pair_cols[f'sma{slow}']) & (pair_cols[f'ma{slow}'] > 0) & (pair_cols[vol_col] < np.quantile(pair_cols[vol_col], vol_q))
    for long_exposure in [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 12.0, 15.0, 20.0]:
        for hedge_exposure in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0]:
            for neutral_exposure in [0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]:
                soxl_exposure = np.where(risk_on, long_exposure, neutral_exposure)
                soxx_exposure = np.where(risk_on, -hedge_exposure, 0.0)
                row = evaluate_pair('soxl_soxx_pair_hedge', soxl_exposure, soxx_exposure)
                if row:
                    row.update({'branch': 'soxl_soxx_pair_hedge', 'fast_window': fast, 'slow_window': slow, 'vol_col': vol_col, 'vol_q': vol_q, 'long_exposure': long_exposure, 'hedge_exposure': hedge_exposure, 'neutral_exposure': neutral_exposure})
                    pair_rows.append(row)
pair_grid = pd.DataFrame(pair_rows)

all_grid = pd.concat([ml_grid, pair_grid], ignore_index=True, sort=False)
all_grid['beats_baseline_return'] = all_grid['return_pct'] > baseline.return_pct
all_grid['passes_drawdown_gate'] = all_grid['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR
all_grid['decision'] = np.where(all_grid['beats_baseline_return'] & all_grid['passes_drawdown_gate'], 'Promote', np.where(all_grid['passes_drawdown_gate'], 'Iterate', 'Drop'))
all_grid['decision_basis'] = np.where(~all_grid['passes_drawdown_gate'], 'drawdown_gate', np.where(~all_grid['beats_baseline_return'], 'return_gate', 'beats_baseline_and_drawdown_gate'))

branch_results = {}
candidate_rows = []
for branch, grid in all_grid.groupby('branch'):
    best_return = grid.sort_values('return_pct', ascending=False).iloc[0]
    drawdown_gate = grid.loc[grid['passes_drawdown_gate']]
    best_drawdown_gate = drawdown_gate.sort_values('return_pct', ascending=False).iloc[0] if not drawdown_gate.empty else None
    passers = grid.loc[grid['beats_baseline_return'] & grid['passes_drawdown_gate']]
    branch_results[branch] = {
        'coverage_start': START,
        'coverage_end': END,
        'grid_rows': int(len(grid)),
        'passes_both_count': int(len(passers)),
        'goal_baseline_return_pct': float(baseline.return_pct),
        'best_return_pct': float(best_return['return_pct']),
        'best_return_max_drawdown_pct': float(best_return['max_drawdown_pct']),
        'best_drawdown_gate_return_pct': float(best_drawdown_gate['return_pct']) if best_drawdown_gate is not None else None,
        'best_drawdown_gate_max_drawdown_pct': float(best_drawdown_gate['max_drawdown_pct']) if best_drawdown_gate is not None else None,
    }
    for label, row in [('best_return', best_return), ('best_drawdown_gate', best_drawdown_gate)]:
        if row is None:
            continue
        candidate_rows.append({'branch': branch, 'candidate': label, 'return_pct': float(row['return_pct']), 'baseline_return_pct': float(baseline.return_pct), 'return_improvement_pct': float(row['return_pct'] - baseline.return_pct), 'max_drawdown_pct': float(row['max_drawdown_pct']), 'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'beats_baseline_return': bool(row['beats_baseline_return']), 'passes_drawdown_gate': bool(row['passes_drawdown_gate']), 'decision': str(row['decision']), 'decision_basis': str(row['decision_basis']), 'grid_rows': int(len(grid))})
candidate_metrics = pd.DataFrame(candidate_rows)
candidate_metrics.to_csv(CARD / 'agentic_alpha_ml_pair_candidate_metrics.csv', index=False)

passes_both = all_grid.loc[all_grid['beats_baseline_return'] & all_grid['passes_drawdown_gate']]
best_candidate = None if passes_both.empty else str(passes_both.sort_values('return_pct', ascending=False).iloc[0]['branch'])
search_summary = {'workflow': 'agentic_alpha_ml_pair_search', 'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'], 'recorder_id': None, 'baseline': {'name': baseline.name, 'return_pct': float(baseline.return_pct), 'max_drawdown_pct': float(baseline.max_drawdown_pct), 'benchmark_return_pct': float(baseline.benchmark_return_pct), 'recorder_id': baseline.recorder_id}, 'goal_achieved': bool(best_candidate is not None), 'best_candidate': best_candidate, 'passes_both_count': int(len(passes_both)), 'branch_results': branch_results, 'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(), 'grid_counts': {'walk_forward_crash_return_ensemble_rows': int(len(ml_grid)), 'soxl_soxx_pair_hedge_rows': int(len(pair_grid))}, 'model_evidence': {'walk_forward': True, 'train_start': features.index.min().date().isoformat(), 'test_start': ml_predictions.index.min().date().isoformat(), 'test_end': ml_predictions.index.max().date().isoformat(), 'model_families': ['LGBMRegressor', 'LGBMClassifier', 'HistGradientBoosting', 'Ridge', 'LogisticRegression'], 'labels': ['next_day_return', 'next_day_loss_lt_minus_6pct', 'next_day_gain_gt_4pct']}}
(CARD / 'agentic_alpha_ml_pair_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'failure_modes.md').write_text('# Failure Modes\n\n- walk_forward_crash_return_ensemble: high-return candidates still carry crash-level drawdowns; drawdown-gated candidates reduce exposure so much that return stays far below the production baseline.\n- soxl_soxx_pair_hedge: SOXX hedges reduce drawdown only when net SOXL exposure is small; return then fails the baseline gate. Higher-return pair candidates remain exposed to the same semiconductor drawdown regime.\n- next_action: use a genuinely new crash-risk signal source or longer intraday/order-flow history; daily SOXL/SOXX/VIX features and SOXX hedges do not close the full objective.\n', encoding='utf-8')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(baseline=baseline.name, provider_uri=PROVIDER_URI.as_posix(), max_drawdown_floor=MAX_DRAWDOWN_FLOOR, ml_grid_rows=len(ml_grid), pair_grid_rows=len(pair_grid), source_url='https://github.com/JacobDu/agentic-alpha')
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_ml_pair_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
    for path in [CARD / 'agentic_alpha_ml_pair_candidate_metrics.csv', CARD / 'agentic_alpha_ml_pair_search_summary.json', CARD / 'external_sources.csv', CARD / 'failure_modes.md']:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_ml_pair')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
search_summary['recorder_id'] = recorder_id
search_summary['recorder_local_dir'] = recorder_local_dir.as_posix()
(CARD / 'agentic_alpha_ml_pair_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

summary = {'experiment': EXPERIMENT, 'card_type': 'diagnostic_search', 'promotion_decision': 'not_promoted', 'as_of': END, 'data_range': [START, END], 'provider_uri': PROVIDER_URI.as_posix(), 'recorder_uri': RECORDER_URI.as_posix(), 'recorder_id': recorder_id, 'recorder_local_dir': recorder_local_dir.as_posix(), 'market': 'all', 'region': 'us', 'baseline': search_summary['baseline'], 'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True}, 'search_report': search_summary, 'branch_results': branch_results, 'external_source_counts': search_summary['external_source_counts'], 'qlib_docs': {'data': 'https://qlib.readthedocs.io/en/latest/component/data.html', 'model': 'https://qlib.readthedocs.io/en/latest/component/model.html', 'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html'}, 'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'], 'workflow_config': 'workflow.yaml', 'sidecars': {'candidate_metrics': 'agentic_alpha_ml_pair_candidate_metrics.csv', 'external_sources': 'external_sources.csv', 'failure_modes': 'failure_modes.md', 'search_summary': 'agentic_alpha_ml_pair_search_summary.json'}, 'artifact_policy': {'policy': 'slim_card_recorder_first', 'workflow_config': 'workflow.yaml', 'machine_entrypoint': 'summary.json', 'primary_artifact': f'{EXPERIMENT}.ipynb', 'kept_sidecars': ['agentic_alpha_ml_pair_candidate_metrics.csv', 'agentic_alpha_ml_pair_search_summary.json', 'external_sources.csv', 'failure_modes.md'], 'recorder_only_artifacts': {'predictions': 'artifacts/pred.pkl', 'labels': 'artifacts/label.pkl', 'signals': 'artifacts/signals.pkl', 'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl', 'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl', 'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl', 'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv'}, 'reproducible_from_recorder': True}}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'conclusion.md').write_text('# 结论\n\n' f'- what_ran: 执行 {EXPERIMENT}，用 qlib `D.features` 读取 SOXL/SOXX/VIX，测试 walk-forward crash/return ensemble 与 SOXL/SOXX pair hedge 两条 R-G-E-V-D 分支。\n' f'- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; 新分支 passes_both_count={len(passes_both)}。\n' '- regime_window_difference: 日线模型和 pair hedge 都能在部分低波趋势窗口提高收益或降低风险，但不能同时保持高收益和 MaxDD<20%。\n' '- promotion_decision: not_promoted，未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。\n' '- risks: 阈值搜索仍依赖日线 SOXL/SOXX/VIX，缺少真正独立的 crash-risk 信息；pair hedge 的 SOXX 对冲会同步削弱 SOXL 牛市收益。\n' '- next_action: 转向新的外部 crash-risk 数据源、期权/波动曲面或更长 intraday/order-flow 特征，不再重复日线 SOXL/SOXX/VIX 门控。\n', encoding='utf-8')

print(json.dumps({'recorder_id': recorder_id, 'passes_both_count': int(len(passes_both)), 'branch_results': branch_results}, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[71815:MainThread](2026-06-23 16:54:04,686) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[71815:MainThread](2026-06-23 16:54:04,688) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[71815:MainThread](2026-06-23 16:54:04,688) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[71815:MainThread](2026-06-23 16:54:34,381) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_ml_pair_search_v1.


[71815:MainThread](2026-06-23 16:54:34,383) INFO - qlib.workflow - [exp.py:258] - Experiment 520348843536460966 starts running ...


[71815:MainThread](2026-06-23 16:54:34,466) INFO - qlib.workflow - [recorder.py:345] - Recorder b5a8b55a20f84e56a02d84bae17b056f starts running under Experiment 520348843536460966 ...


[71815:MainThread](2026-06-23 16:54:34,564) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{
  "branch_results": {
    "soxl_soxx_pair_hedge": {
      "best_drawdown_gate_max_drawdown_pct": -19.895618308143213,
      "best_drawdown_gate_return_pct": 317.87162704180776,
      "best_return_max_drawdown_pct": -76.12638644024862,
      "best_return_pct": 15268.442999024322,
      "coverage_end": "2026-06-09",
      "coverage_start": "2019-01-02",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 6391,
      "passes_both_count": 0
    },
    "walk_forward_crash_return_ensemble": {
      "best_drawdown_gate_max_drawdown_pct": -16.68242233388345,
      "best_drawdown_gate_return_pct": 267.9795331485608,
      "best_return_max_drawdown_pct": -91.69888554997115,
      "best_return_pct": 9277.572046667008,
      "coverage_end": "2026-06-09",
      "coverage_start": "2019-01-02",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 27734,
      "passes_both_count": 0
    }
  },
  "passes_both_count": 0,
  "recorder_id": "b5a8b55a20f84e56a02d84bae17b